# SynBioCrow 2.3 — BioPKS Resumable Direct Beam Search\nCheckpointed, restart-safe direct enzymatic search. Uses a narrow beam (12), caches expensive chemistry objects, persists after every expanded node, and resumes from Drive.\n

In [ ]:
import os,sys,subprocess,shutil,json,zipfile,hashlib
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_biopks_resumable")
BIOPKS_ROOT=Path("/content/BioPKS-Pipeline")
VENV=Path("/content/biopks_py310_resumable")
PY=VENV/"bin/python"
GATE=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/biopks_reentry")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/biopks_resumable_beam")
OUT.mkdir(parents=True,exist_ok=True)

def run(cmd,*,cwd=None,timeout=1800,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
run(["git","clone","--depth","1","https://github.com/JBEI/BioPKS-Pipeline.git",str(BIOPKS_ROOT)],timeout=600)
run([sys.executable,"-m","pip","install","-q","uv"],timeout=600)
run(["uv","python","install","3.10"],timeout=900)
if VENV.exists(): shutil.rmtree(VENV)
run(["uv","venv","--python","3.10",str(VENV)],timeout=600)

req=BIOPKS_ROOT/"requirements.txt"
patched=BIOPKS_ROOT/"requirements.synbiocrow_colab.txt"
patched.write_text("\n".join(x for x in req.read_text().splitlines() if x.strip().lower()!="umap==0.1.1")+"\n")
run(["uv","pip","install","--python",str(PY),"-r",str(patched)],timeout=1800)
run(["uv","pip","install","--python",str(PY),"setuptools<81"],timeout=600)
run(["uv","pip","install","--python",str(PY),"doranet==0.5.7a1"],timeout=600)
run(["uv","pip","install","--python",str(PY),"--no-deps","-e",str(BIOPKS_ROOT)],timeout=900)

panel=json.loads((REPO/"benchmarks/2.3/biopks_reentry_panel.json").read_text())
targets={x["target_id"]:x for x in panel["targets"]}
search=REPO/"scripts/search_biopks_resumable_beam_2_3.py"

terminal={"EXACT_HIT","NO_EXACT_HIT_WITHIN_BOUNDS"}
summary={"schema":"synbiocrow.biopks_resumable_beam_summary.v1","targets":{}}

for tid in ("pks_naringenin","pks_pinocembrin"):
    saved=GATE/f"{tid}.pks_only.json"
    gate=json.loads(saved.read_text())
    routes=(gate.get("response") or {}).get("routes") or []
    top=max(routes,key=lambda x:float(x.get("score") or -1))
    precursor=top["predicted_product_smiles"]
    target=targets[tid]["target_smiles"]

    state=OUT/f"{tid}.state.json"
    result=OUT/f"{tid}.result.json"

    # Autonomous resumable chunks. Each chunk checkpoints after every expanded node.
    for chunk in range(1,13):
        print(f"\n[{tid}] CHUNK {chunk}")
        cmd=[str(PY),str(search),
             "--precursor",precursor,
             "--target",target,
             "--target-name",tid,
             "--ruleset","JN1224MIN",
             "--max-depth","3",
             "--beam-width","12",
             "--max-products-per-rule","8",
             "--time-budget","240",
             "--state-file",str(state),
             "--output",str(result)]
        run(cmd,cwd=OUT,timeout=300,check=False)
        payload=json.loads(result.read_text())
        print("STATUS",payload["status"],
              "depth",payload["depth"],
              "node",payload["node_index"],"/",payload["frontier_size"],
              "best",payload["best_similarity"])
        if payload["status"] in terminal:
            break
        if state.is_file():
            st=json.loads(state.read_text())
            if st.get("status")=="PAUSED_BUDGET":
                st["status"]="RUNNING"
                state.write_text(json.dumps(st,indent=2,sort_keys=True)+"\n")

    summary["targets"][tid]=json.loads(result.read_text()) if result.is_file() else {"status":"NO_RESULT"}

report=OUT/"biopks_resumable_beam_summary.json"
report.write_text(json.dumps(summary,indent=2,sort_keys=True)+"\n")
print("\nSUMMARY")
for tid,r in summary["targets"].items():
    print(tid,r.get("status"),"best=",r.get("best_similarity"),"levels=",r.get("levels"))

bundle=OUT/"SynBioCrow_2_3_BIOPKS_RESUMABLE_BEAM_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*.json")):
        z.write(p,p.name)
    z.write(search,search.name)
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
